# Dopamine PINN: baseline reproduction

Regenerates **every number in the manuscript** (Hackman & Zhu, MBEC, v5 and later)
through one code path, the `dopamine_pinn` package of this repository, from the frozen
benchmark data in `benchmark/v1/`. It also measures the run-to-run variation of the
PINN (network seed, collocation draw, GPU non-determinism), which later studies need to
tell a real improvement from noise.

| Toggle | Manuscript item | PINN runs | A100 time (est.) |
|---|---|---|---|
| `RUN_FORWARD` | forward accuracy (Table 2, Figs 2-3) and architecture ablation (OR 4) | 3 | ~15 min |
| `RUN_CANONICAL` | Table 3 upper rows, Laplace posterior (Fig 6), LM baseline (Table 4), trajectories (OR 4) | 5 | ~17 min |
| `RUN_VARIATION` | run-to-run variation (new) | 5 | ~17 min |
| `RUN_NOISE` | noise sensitivity (Fig 5, OR 4) | 5 | ~17 min |
| `RUN_GRID` | parameter grid (OR 4) | 5 | ~17 min |
| `RUN_DENSITY` | number of observations (OR 4) | 5 | ~17 min |
| `RUN_DRAWS` | 20 realizations (Table 3 lower rows, Fig 4, OR 4) | 20 | ~1.1 h |
| `RUN_TRUTHFREE` | data-weight selection without the truth (Section 3.4, OR 3) | 42 | ~2.3 h |

Results are saved after every run (to Google Drive with `USE_DRIVE = True`) and a rerun
skips finished runs, so the experiments can be spread over several sessions. The last
cells collect everything into `paper_numbers.json` and download a zip.

## 0. Install (Colab): pinned JAX stack + this repository as a package

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/Hachero98/dopamine-PINN-.git'
REPO_REF = 'fix/literature-k-parameter'   # branch or release tag
REPO_DIR = Path('/content/dopamine-PINN-')
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
SENTINEL = Path('/tmp/dopamine_pinn_ready_v1')

if IN_COLAB and not SENTINEL.exists():
    rc = os.system('pip -q install "jax[cuda12]==0.10.2" "flax==0.12.5" "optax>=0.2.3" "jaxopt>=0.8" scipy matplotlib')
    if not REPO_DIR.exists():
        rc |= os.system(f'git clone -q -b {REPO_REF} {REPO_URL} {REPO_DIR}')
    rc |= os.system(f'pip -q install --no-deps -e {REPO_DIR}')
    if rc != 0:
        raise RuntimeError('install failed - see the output above')
    SENTINEL.touch()
    print('Installed. Restarting the kernel; then Runtime -> Run all again.')
    os.kill(os.getpid(), 9)
else:
    print('Ready.')

In [ ]:
import json, time
import numpy as np
import jax, jax.numpy as jnp
import matplotlib.pyplot as plt

import dopamine_pinn as dp
from dopamine_pinn import config as cfg, benchmark
from dopamine_pinn.solver import make_fd_solver, solve_at, linear_sink, interp_frames, analytical_infinite
from dopamine_pinn.reference import fit_linear, lm_analytical, laplace_interval
from dopamine_pinn.pinn import TrainConfig, predict
from dopamine_pinn import experiments as ex

print(f'dopamine_pinn {dp.__version__}  |  JAX {jax.__version__}  |  devices {jax.devices()}')
print(f'benchmark data: {benchmark.DEFAULT_DIR} ({len(benchmark.names())} datasets)')

## 1. Toggles

In [ ]:
QUICK         = False   # tiny schedule for a smoke test (not results)
USE_DRIVE     = True
RUN_FORWARD   = True
RUN_CANONICAL = True
RUN_VARIATION = True
RUN_NOISE     = True
RUN_GRID      = True
RUN_DENSITY   = True
RUN_DRAWS     = True
RUN_TRUTHFREE = True

TC = TrainConfig.quick(verbose=True, log_every=4) if QUICK else TrainConfig()
NET_SEEDS = [1234, 1235, 1236, 1237, 1238]
PTS_SEED = cfg.SEED                     # the paper draws one collocation set (seed 1234)

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/dopamine_PINN_baseline')
else:
    OUT = Path('baseline_out')
if QUICK:
    OUT = OUT / 'quick'
OUT.mkdir(parents=True, exist_ok=True)
print(f'Results -> {OUT.resolve()}  |  Adam {TC.adam_iters}, L-BFGS {TC.lbfgs_iters}')


def store(name):
    """Load-or-create a results file; returns (dict, save function)."""
    path = OUT / f'{name}.json'
    res = json.loads(path.read_text()) if path.exists() else {}
    def save():
        path.write_text(json.dumps(res, indent=1, default=float))
    return res, save


def public(d):
    return {k: v for k, v in d.items() if not k.startswith('_')}


SOLVER = make_fd_solver()

## 2. Forward problem and architecture ablation

In [ ]:
if RUN_FORWARD:
    res, save = store('B_forward')
    archs = {'canonical': cfg.LAYERS, 'compact': (3, 32, 32, 32, 1), 'wide': (3, 128, 128, 128, 1)}
    tc_f = TC.replace(weights=cfg.W_FORWARD)
    for name, layers in archs.items():
        if name in res:
            continue
        r = ex.forward(tc_f, layers=layers)
        res[name] = public(r)
        if name == 'canonical':      # data for Figs 2 and 3
            _, fr = ex.truth_frames()
            xs = np.linspace(-cfg.L / 2, cfg.L / 2, 201)
            prof = {}
            for t_s in (1.0, 5.0, 50.0):
                prof[f'pinn_t{t_s:g}'] = predict(r['_net'], r['_p'], xs, 0 * xs, np.full_like(xs, t_s))
                prof[f'fd_t{t_s:g}'] = np.asarray(interp_frames(fr, SOLVER.dx, jnp.asarray(xs), jnp.zeros_like(jnp.asarray(xs)),
                                                                jnp.full(xs.shape, t_s)))
                prof[f'an_t{t_s:g}'] = analytical_infinite(xs, 0 * xs, t_s)
            xh = np.linspace(-cfg.L / 2, cfg.L / 2, 101)
            Xh, Yh = np.meshgrid(xh, xh, indexing='ij')
            prof['field_pinn_t5'] = predict(r['_net'], r['_p'], Xh, Yh, np.full_like(Xh, 5.0))
            prof['field_fd_t5'] = np.asarray(interp_frames(fr, SOLVER.dx, jnp.asarray(Xh.ravel()), jnp.asarray(Yh.ravel()),
                                                           jnp.full(Xh.size, 5.0))).reshape(Xh.shape)
            np.savez(OUT / 'B_forward_profiles.npz', x=xs, xh=xh, **prof)
        save()
        print(f'  forward {name}: L2 vs FD {r["L2_vs_fd"]:.2f}%  ' +
              ', '.join(f'{k} {v["vs_fd"]:.2f}%' for k, v in r['snapshots'].items()) + f'  ({r["minutes"]:.1f} min)')

## 3. Canonical realization 4001: five network seeds, reference estimator, Laplace, LM

In [ ]:
if RUN_CANONICAL:
    res, save = store('B_canonical')
    obs, meta = benchmark.load('linear_s4001')
    if 'reference' not in res:
        t0 = time.time()
        ref = fit_linear(obs, SOLVER)
        lo_D, hi_D = laplace_interval(ref['D'], ref['sd_logD'])
        lo_k, hi_k = laplace_interval(ref['k'], ref['sd_logk'])
        res['reference'] = {'D': ref['D'], 'k': ref['k'], 'D_err': ex.pct(ref['D'], cfg.D_TRUE),
                            'k_err': ex.pct(ref['k'], cfg.K_TRUE), 'sd_logD': ref['sd_logD'], 'sd_logk': ref['sd_logk'],
                            'rho': ref['rho'], 'D_95': [lo_D, hi_D], 'k_95': [lo_k, hi_k],
                            'cov_log': ref['fit']['cov'].tolist(), 'seconds': time.time() - t0}
        t0 = time.time()
        lm = lm_analytical(obs)
        res['lm'] = {**lm, 'D_err': ex.pct(lm['D'], cfg.D_TRUE), 'k_err': ex.pct(lm['k'], cfg.K_TRUE),
                     'milliseconds': 1000 * (time.time() - t0)}
        save()
    for s in NET_SEEDS:
        key = f'net{s}'
        if key in res:
            continue
        r = ex.inverse(obs, TC, seed=s, pts_seed=PTS_SEED, label=f'canonical net {s}')
        res[key] = {**ex.linear_errors(r), 'L': r['L'], 'minutes': r['minutes'], 'trajectory': r['trajectory']}
        save()
        print(f'  net seed {s}: D {res[key]["D_err"]:+.2f}%  k {res[key]["k_err"]:+.2f}%  ({r["minutes"]:.1f} min)')
    runs = [res[f'net{s}'] for s in NET_SEEDS if f'net{s}' in res]
    absD, absk = [abs(r['D_err']) for r in runs], [abs(r['k_err']) for r in runs]
    print(f'\n  PINN over {len(runs)} seeds: D {np.mean([r["D"] for r in runs]):.4f} +- {np.std([r["D"] for r in runs], ddof=1) if len(runs) > 1 else 0:.4f}'
          f' (|err| {np.mean(absD):.2f} +- {np.std(absD, ddof=1) if len(runs) > 1 else 0:.2f}%),'
          f' k {np.mean([r["k"] for r in runs]):.5f} (|err| {np.mean(absk):.2f}%)')
    print(f'  reference: D {res["reference"]["D"]:.4f} ({res["reference"]["D_err"]:+.2f}%), '
          f'k {res["reference"]["k"]:.5f} ({res["reference"]["k_err"]:+.2f}%), rho {res["reference"]["rho"]:+.3f}')
    print(f'  LM (analytical): D {res["lm"]["D"]:.4f}, k {res["lm"]["k"]:.5f}')

## 4. Run-to-run variation on realization 4001

Three sources, each varied alone around the configuration (network seed 1234,
collocation seed 1234): network initialization (the five canonical seeds), the
collocation draw (seeds 1235-1238 with network seed 1234), and GPU non-determinism (the
configuration (1234, 1234) trained a second time).

In [ ]:
if RUN_VARIATION:
    res, save = store('B_variation')
    obs, _ = benchmark.load('linear_s4001')
    jobs = [(f'pts{s}', 1234, s) for s in (1235, 1236, 1237, 1238)] + [('repeat', 1234, 1234)]
    for key, net_s, pts_s in jobs:
        if key in res:
            continue
        r = ex.inverse(obs, TC, seed=net_s, pts_seed=pts_s, label=f'variation {key}')
        res[key] = {**ex.linear_errors(r), 'net_seed': net_s, 'pts_seed': pts_s, 'minutes': r['minutes']}
        save()
        print(f'  {key}: D {res[key]["D_err"]:+.2f}%  k {res[key]["k_err"]:+.2f}%')
    can, _ = store('B_canonical')
    if all(f'net{s}' in can for s in NET_SEEDS) and all(k in res for k, _, _ in jobs):
        sd = lambda v: float(np.std(v, ddof=1))
        net_D = [can[f'net{s}']['D_err'] for s in NET_SEEDS]
        net_k = [can[f'net{s}']['k_err'] for s in NET_SEEDS]
        pts_keys = [k for k, _, _ in jobs if k.startswith('pts')]
        pts_D = [can['net1234']['D_err']] + [res[k]['D_err'] for k in pts_keys]
        pts_k = [can['net1234']['k_err']] + [res[k]['k_err'] for k in pts_keys]
        res['summary'] = {'sd_network_seed': {'D': sd(net_D), 'k': sd(net_k)},
                          'sd_collocation_seed': {'D': sd(pts_D), 'k': sd(pts_k)},
                          'repeat_difference': {'D': res['repeat']['D_err'] - can['net1234']['D_err'],
                                                'k': res['repeat']['k_err'] - can['net1234']['k_err']}}
        save()
        print('  SD of the signed error (percentage points):', json.dumps(res['summary'], indent=1))

## 5. Noise sensitivity, parameter grid and number of observations

In [ ]:
def sweep(file, datasets, label):
    res, save = store(file)
    for name in datasets:
        if name in res:
            continue
        obs, meta = benchmark.load(name)
        noise = meta['noise_sd']
        out, r = ex.inverse_with_reference(obs, SOLVER, TC, seed=cfg.SEED, pts_seed=PTS_SEED, noise_sd=noise,
                                           D_true=meta['D'], k_true=meta['k'], label=f'{label} {name}')
        res[name] = {**out, 'meta': {k: meta[k] for k in ('D', 'k', 'noise_sd', 'n_obs')}}
        save()
        print(f'  {name}: PINN D {out["D_err"]:+.2f}% k {out["k_err"]:+.2f}% | ref D {out["ref_D_err"]:+.2f}% '
              f'k {out["ref_k_err"]:+.2f}%  ({out["minutes"]:.1f} min)')
    return res


if RUN_NOISE:
    sweep('B_noise', [f'linear_noise{e:g}_s4001' for e in (0, 1, 2, 5, 10)], 'noise')
if RUN_GRID:
    sweep('B_grid', [f'linear_grid_D{D:g}_k{k:g}_s4001' for D, k in
                     ((0.16, 0.010), (0.16, 0.040), (0.32, 0.020), (0.50, 0.010), (0.50, 0.040))], 'grid')
if RUN_DENSITY:
    sweep('B_density', [f'linear_n{n}_s4001' for n in (100, 200, 400, 800, 1600)], 'density')

## 6. Twenty independent realizations (seeds 7001-7020)

In [ ]:
if RUN_DRAWS:
    from scipy.stats import chi2
    res = sweep('B_draws', [f'linear_s{s}' for s in range(7001, 7021)], 'draw')
    R = [res[f'linear_s{s}'] for s in range(7001, 7021) if f'linear_s{s}' in res]
    A = lambda k: np.array([r[k] for r in R])
    def inside(est_D, est_k, cen_D, cen_k, r):
        cov = np.diag([r['ref_sd_logD']**2, r['ref_sd_logk']**2])
        cov[0, 1] = cov[1, 0] = r['ref_rho'] * r['ref_sd_logD'] * r['ref_sd_logk']
        v = np.log([est_D / cen_D, est_k / cen_k])
        return bool(v @ np.linalg.solve(cov, v) <= chi2.ppf(0.95, 2))
    summ = {}
    for m in ('', 'ref_'):
        for q in ('D', 'k'):
            e = A(f'{m}{q}_err')
            summ[f'{m or "pinn_"}{q}'] = {'mean': e.mean(), 'sd': e.std(ddof=1) if e.size > 1 else 0,
                                          'rmse': float(np.sqrt(np.mean(e**2)))}
    summ['ref_cover_D'] = float(np.mean([abs(np.log(r['ref_D'] / cfg.D_TRUE)) <= 1.96 * r['ref_sd_logD'] for r in R]))
    summ['ref_cover_k'] = float(np.mean([abs(np.log(r['ref_k'] / cfg.K_TRUE)) <= 1.96 * r['ref_sd_logk'] for r in R]))
    summ['ref_cover_joint'] = float(np.mean([inside(cfg.D_TRUE, cfg.K_TRUE, r['ref_D'], r['ref_k'], r) for r in R]))
    summ['pinn_in_ref95_joint'] = float(np.mean([inside(r['D'], r['k'], r['ref_D'], r['ref_k'], r) for r in R]))
    summ['mean_laplace_sd_pct'] = {'D': 100 * A('ref_sd_logD').mean(), 'k': 100 * A('ref_sd_logk').mean()}
    if len(R) > 2:
        summ['corr_pinn_ref'] = {'D': float(np.corrcoef(A('D_err'), A('ref_D_err'))[0, 1]),
                                 'k': float(np.corrcoef(A('k_err'), A('ref_k_err'))[0, 1])}
        summ['mean_pinn_minus_ref_D'] = float(np.mean(A('D_err') - A('ref_D_err')))
    res['summary'] = summ
    (OUT / 'B_draws.json').write_text(json.dumps(res, indent=1, default=float))
    print(json.dumps(summ, indent=1, default=float))

## 7. Data weight without the ground truth (discrepancy principle, 4-fold cross-validation)

In [ ]:
W_D_LIST = [1.0, 3.0, 5.0, 10.0, 30.0, 100.0]
if RUN_TRUTHFREE:
    res, save = store('B_truthfree')
    sig = cfg.NOISE_SD
    for s in (4001, 8001, 8002):
        obs, _ = benchmark.load(f'linear_s{s}')
        for w in W_D_LIST:
            key = f'full_s{s}_w{w:g}'
            if key in res:
                continue
            r = ex.inverse(obs, TC.replace(weights=cfg.W_INVERSE[:3] + (w,)), seed=cfg.SEED, pts_seed=PTS_SEED, label=key)
            res[key] = {**ex.linear_errors(r), 'w_d': w, 'seed': s, 'rms_over_sigma': r['rms_data'] / sig,
                        'minutes': r['minutes']}
            save()
            print(f'  {key}: D {res[key]["D_err"]:+.2f}% k {res[key]["k_err"]:+.2f}% RMS/sigma {res[key]["rms_over_sigma"]:.3f}')
    obs, _ = benchmark.load('linear_s4001')
    fold = np.random.default_rng(99).permutation(len(obs['C_d'])) % 4
    for w in W_D_LIST:
        for f in range(4):
            key = f'cv_w{w:g}_f{f}'
            if key in res:
                continue
            tr = {k: v[fold != f] for k, v in obs.items()}
            te = {k: v[fold == f] for k, v in obs.items()}
            r = ex.inverse(tr, TC.replace(weights=cfg.W_INVERSE[:3] + (w,)), seed=cfg.SEED, pts_seed=PTS_SEED, label=key)
            pred = predict(r['_net'], r['_p'], te['x_d'], te['y_d'], te['t_d'])
            res[key] = {**ex.linear_errors(r), 'w_d': w, 'fold': f,
                        'heldout_mse_over_sigma2': float(np.mean((pred - te['C_d'])**2) / sig**2), 'minutes': r['minutes']}
            save()
            print(f'  {key}: held-out MSE / sigma^2 = {res[key]["heldout_mse_over_sigma2"]:.3f}')
    cv = {w: [res[f'cv_w{w:g}_f{f}']['heldout_mse_over_sigma2'] for f in range(4) if f'cv_w{w:g}_f{f}' in res]
          for w in W_D_LIST}
    sel = {'cv_mean': {str(w): float(np.mean(v)) for w, v in cv.items() if v},
           'cv_se': {str(w): float(np.std(v, ddof=1) / np.sqrt(len(v))) for w, v in cv.items() if len(v) > 1}}
    if sel['cv_mean']:
        sel['cv_choice'] = float(min(sel['cv_mean'], key=sel['cv_mean'].get))
    disc = {}
    for s in (4001, 8001, 8002):
        rows = [res[f'full_s{s}_w{w:g}'] for w in W_D_LIST if f'full_s{s}_w{w:g}' in res]
        ok = [r for r in rows if r['rms_over_sigma'] <= 1.0]
        if rows:
            disc[str(s)] = (min(ok, key=lambda r: r['w_d']) if ok else max(rows, key=lambda r: r['w_d']))['w_d']
    sel['discrepancy_choice'] = disc
    res['selection'] = sel
    save()
    print(json.dumps(sel, indent=1))

## 8. Collect the manuscript numbers

In [ ]:
paper = {'package_version': dp.__version__, 'jax': jax.__version__, 'device': str(jax.devices()[0]),
         'train_config': {k: getattr(TC, k) for k in ('adam_iters', 'lbfgs_iters', 'lr', 'weights')}}
for name in ('B_forward', 'B_canonical', 'B_variation', 'B_noise', 'B_grid', 'B_density', 'B_draws', 'B_truthfree'):
    p = OUT / f'{name}.json'
    if p.exists():
        d = json.loads(p.read_text())
        paper[name[2:]] = {k: v for k, v in d.items() if k in ('summary', 'selection', 'reference', 'lm')} or None
if (OUT / 'B_forward.json').exists():
    f = json.loads((OUT / 'B_forward.json').read_text())
    paper['forward'] = {k: {'L2_vs_fd': v['L2_vs_fd'], 'snapshots': v['snapshots'], 'n_params': v['n_params'],
                            'minutes': v['minutes']} for k, v in f.items()}
if (OUT / 'B_canonical.json').exists():
    c = json.loads((OUT / 'B_canonical.json').read_text())
    runs = [c[f'net{s}'] for s in NET_SEEDS if f'net{s}' in c]
    if runs:
        paper['canonical']['pinn_5_seeds'] = {
            'D_mean': float(np.mean([r['D'] for r in runs])), 'D_sd': float(np.std([r['D'] for r in runs], ddof=1)) if len(runs) > 1 else 0,
            'abs_D_err_mean': float(np.mean([abs(r['D_err']) for r in runs])),
            'k_mean': float(np.mean([r['k'] for r in runs])), 'k_sd': float(np.std([r['k'] for r in runs], ddof=1)) if len(runs) > 1 else 0,
            'abs_k_err_mean': float(np.mean([abs(r['k_err']) for r in runs]))}
(OUT / 'paper_numbers.json').write_text(json.dumps(paper, indent=1, default=float))
print(json.dumps(paper, indent=1, default=float)[:4000])

In [ ]:
import shutil
zip_path = shutil.make_archive(str(OUT.parent / 'dopamine_PINN_baseline_results'), 'zip', OUT)
print('Zipped:', zip_path)
try:
    from google.colab import files as colab_files
    colab_files.download(zip_path)
except ImportError:
    pass